# PyTorch

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

In [ ]:
REBUILD_DATA = False

class DogsVSCats():
  IMG_SIZE = 50
  CATS = "/content/drive/My Drive/CatsAndDogs/PetImages/Cat"
  DOGS = "/content/drive/My Drive/CatsAndDogs/PetImages/Dog"
  LABELS = {CATS: 0, DOGS: 1}
  training_data = []
  
  catcount = 0
  dogcount = 0
  
  def make_training_data(self):
    for label in self.LABELS:
      for f in tqdm(os.listdir(label)):
        if "jpg" in f:
          try:
            path = os.path.join(label, f)
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (self.IMG_SIZE, self.IMG_SIZE))
            self.training_data.append([np.array(img), np.eye(2)[self.LABELS[label]]])
            
            if label == self.CATS:
              self.catcount += 1
            if label == self.DOGS:
              self.dogcount += 1
              
          except Exception as e:
            pass  
          
    np.random.shuffle(self.training_data)
    np.save("training_data.npy", self.training_data)
    print("Cats:", self.catcount)
    print("dogs:", self.dogcount)
  
if REBUILD_DATA:
  dogsvscats = DogsVSCats()
  dogsvscats.make_training_data()
  
  

In [ ]:
training_data = np.load("training_data.py", allow_pickle=True)
print(len(training_data))

In [ ]:
import torch

X = torch.Tensor([i[0] for i in training_data]).view(-1, 50, 50)
X = X/255.0
y = torch.Tensor([i[1] for i in training_data])

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(X[0], cmap = "gray")

In [ ]:
print(y[0])

# TensorFlow

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import cv2
import random
import pickle
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Flatten, Conv2D, MaxPooling2D

In [ ]:
DATADIR = "/content/drive/My Drive/CatsAndDogs/PetImages"
CATEGORIES = ["Dog", "Cat"]

In [ ]:
training_data = []
IMG_SIZE = 100

def create_training_data():

  for category in CATEGORIES:
    path = os.path.join(DATADIR, category)
    class_num = CATEGORIES.index(category)
    print("Now Executing:", category)

    for cnt, img in enumerate(os.listdir(path)):
      try:
        img_arr = cv2.imread(os.path.join(path, img), cv2.IMREAD_GRAYSCALE)
        new_array = cv2.resize(img_arr, (IMG_SIZE, IMG_SIZE))
        training_data.append([new_array, class_num])
      except Exception as e:
        pass
      print('=', end = '')
      if (cnt > 0) and (cnt%99 == 0):
        print('\n')
        print("Completed Reading {} {} images".format(cnt+1, category))
        print("Length of Training Data: ", len(training_data))
        print('\n')

# create_training_data()

# print(len(training_data))

# random.shuffle(training_data)

# X = []
# Y = []

# for features, label in training_data:
#   X.append(features)
#   Y.append(label)

# pickle_out = open("X.pickle","wb")
# pickle.dump(X, pickle_out)
# pickle_out.close()

# pickle_out = open("Y.pickle","wb")
# pickle.dump(Y, pickle_out)
# pickle_out.close()

In [ ]:
pickle_in = open("/content/drive/My Drive/CatsAndDogs/PetImages/X.pickle","rb")
X = pickle.load(pickle_in)

pickle_in = open("/content/drive/My Drive/CatsAndDogs/PetImages/y.pickle","rb")
y = pickle.load(pickle_in)

In [ ]:
X = np.array(X).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
y = np.array(y)

In [ ]:
X = X/255.0

In [ ]:
model = Sequential()

In [ ]:
model.add(Conv2D(256, (3, 3), input_shape = X.shape[1:]))
model.add(Activation('relu'))
model.add(MaxPooling2D((2, 2)))

model.add(Conv2D(256, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Flatten())
model.add(Dense(1))
model.add(Activation('sigmoid'))

In [ ]:
model.compile(optimizer='adam', loss = 'binary_crossentropy', metrics=['accuracy'])

In [ ]:
model.fit(X, y, batch_size=32, epochs = 10)